# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nazama-tech/Flyrank-Ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [2]:
import os, sys, subprocess
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


My Lane: Rank Signal Analysis

My Why

I'm picking this because visibility is the first stage in the funnel, a page can't get clicked or read if it never gets shown in search results in the first place, so understanding what drives visibility comes before understanding clicks or engagement

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Decision: The content strategist decides what to prioritize when briefing or planning new content, which topics, formats, or search intents are worth investing writer time in, based on patterns historically associated with getting found in search at all.

Action: Before greenlighting a content brief, the strategist checks whether the proposed topic/intent matches the patterns that correlate with visibility rather than guessing based on instinct alone.

Cost of a wrong call: A wrong call here doesn't cost one page's traffic the way a missed refresh would it costs repeated, compounding misallocation of writer time. If the strategist trusts a correlation that isn't actually real, they could steer multiple future briefs toward the wrong format or topic before anyone realizes the underlying signal was never solid

In [5]:
# This cell is for CODE (numbers, a query, a check).
print("Median visibility by content_type:")
print(df.groupby("content_type")["impressions_90d"].median().sort_values(ascending=False))
print()
print("Median visibility by main_intent:")
print(df.groupby("main_intent")["impressions_90d"].median().sort_values(ascending=False))

Median visibility by content_type:
content_type
keyword article       955.0
comparison article    107.0
feedly article          4.0
Name: impressions_90d, dtype: float64

Median visibility by main_intent:
main_intent
transactional    1070.0
commercial        990.0
informational     848.0
navigational       93.0
Name: impressions_90d, dtype: float64


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three numbers make this lane worth the next 7 weeks. First, word_count shows a real, moderate correlation with visibility, a usable signal, not noise. Second, search_volume, which you'd naively expect to predict visibility, showing that demand size doesn't guarantee a page captures that demand; a real explanation needs more than one variable. Third, the data itself has a trap: rows with avg_position == 0 have a median of just 1 impression, while genuine top-3 pages median 74, meaning a naive analysis would have concluded position doesn't matter, when the opposite is true. That combination, real signal, a genuine surprise, and a data-quality trap, is exactly the kind of ground worth spending 7 weeks on rather than a quick glance

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Ranking signal analysis
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id")
df["log_impressions"] = np.log1p(df["impressions_90d"])

print("Corr(search_volume, log visibility):", round(df["search_volume"].corr(df["log_impressions"]), 3))

real_top3 = df[(df["avg_position"] > 0) & (df["avg_position"] <= 3)]
zero_pos  = df[df["avg_position"] == 0]
print("Median impressions, real top-3:", real_top3["impressions_90d"].median())
print("Median impressions, placeholder rows (avg_position==0):", zero_pos["impressions_90d"].median())


Corr(search_volume, log visibility): 0.004
Median impressions, real top-3: 74.0
Median impressions, placeholder rows (avg_position==0): 1.0


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

What this work CAN say: Findings are observed, associational, and directional, e.g., "pages with higher word count are associated with higher visibility." These can support a content strategist's decisions the way a lab trend narrows a differential, without confirming it.

What this work CANNOT say: It cannot claim causation. Pages weren't randomly assigned their word count, content type, or intent, a longer page might also happen to target an easier keyword, or be written by a more experienced writer, or cover a naturally broader topic. Any of those hidden factors could be doing the real work, the same way a chart review can't prove Drug X caused an outcome when patients weren't randomly assigned to receive it, some other factor (a second treatment, disease severity, access to care) could be the real driver. This work also cannot predict or reverse-engineer Google's actual ranking algorithm; it only describes outcomes observed on this site's own data.

In [9]:
#Sanity check: are our "signal" variables even independent of each other,
# or could they be tangled together (a confounding risk)?
print(df[["word_count","search_volume","competition"]].corr())

               word_count  search_volume  competition
word_count       1.000000      -0.019458    -0.201019
search_volume   -0.019458       1.000000     0.049887
competition     -0.201019       0.049887     1.000000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.